In [1]:
import pandas as pd

df = pd.read_csv('../data/books.csv')

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
df.head(3)

Shape: (6810, 12)

Columns: ['isbn13', 'isbn10', 'title', 'subtitle', 'authors', 'categories', 'thumbnail', 'description', 'published_year', 'average_rating', 'num_pages', 'ratings_count']


,isbn13,isbn10,title,subtitle,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count
0,9780002005883,0002005883,Gilead,NaN,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0
1,9780002261982,0002261987,Spider's Web,A Novel,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0
2,9780006163831,0006163831,The One Tree,NaN,Stephen R. Donaldson,American fiction,http://books.google.com/books/content?id=OmQaw...,Volume Two of Stephen Donaldson's acclaimed se...,1982.0,3.97,479.0,172.0


In [2]:
print(df.isnull().sum())

isbn13               0
isbn10               0
title                0
subtitle          4429
authors             72
categories          99
thumbnail          329
description        262
published_year       6
average_rating      43
num_pages           43
ratings_count       43
dtype: int64


In [3]:
print(df['description'][0])

A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gilead is a song of celebration and acceptance of the best and the worst the world has

In [4]:
# Keep only columns useful for our project
columns_to_keep = [
    'title', 
    'authors', 
    'categories', 
    'description', 
    'average_rating', 
    'thumbnail',
    'published_year'
]

df = df[columns_to_keep]
print("After column drop:", df.shape)

After column drop: (6810, 7)


In [5]:
# Description is our most important column
# Books without it are useless for embedding
df = df.dropna(subset=['description'])
print("After dropping missing descriptions:", df.shape)

After dropping missing descriptions: (6548, 7)


In [6]:
# For other columns, fill missing with sensible defaults
# We don't drop these rows — we just fill the gaps
df['authors'] = df['authors'].fillna('Unknown Author')
df['categories'] = df['categories'].fillna('Uncategorized')
df['average_rating'] = df['average_rating'].fillna(0.0)
df['published_year'] = df['published_year'].fillna(0)
df['thumbnail'] = df['thumbnail'].fillna('')

print("Missing values after cleaning:")
print(df.isnull().sum())

Missing values after cleaning:
title             0
authors           0
categories        0
description       0
average_rating    0
thumbnail         0
published_year    0
dtype: int64


In [7]:

df['description'] = df['description'].str.strip()

df = df[df['description'].str.split().str.len() >= 20]

print("After removing short descriptions:", df.shape)

After removing short descriptions: (5729, 7)


In [8]:

df['text_to_embed'] = (
    "Title: " + df['title'] + ". " +
    "Author: " + df['authors'] + ". " +
    "Category: " + df['categories'] + ". " +
    "Description: " + df['description']
)

# Check what it looks like
print(df['text_to_embed'][0])

Title: Gilead. Author: Marilynne Robinson. Category: Fiction. Description: A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gilead is a

In [9]:
df = df.reset_index(drop=True)
df.to_csv('../data/cleaned_books.csv', index=False)

print("✅ Cleaned data saved!")
print("Final shape:", df.shape)

✅ Cleaned data saved!
Final shape: (5729, 8)
